In [2]:
!pip install -q "transformers==4.51.3" "datasets>=2.20.0" "accelerate>=0.33.0" sentencepiece safetensors scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 96.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 105.1 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import json
import os
import random
import re
import shutil
from collections import Counter, defaultdict

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab or Drive already mounted:', exc)

shortcut_path = Path('/content/drive/MyDrive/til26_data')

def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / 'nlp').exists() and ((path / 'data' / 'nlp').exists() or (path / 'CODEX.md').exists()):
            return path
    candidate = Path('/content/til-26-overflow')
    return candidate if candidate.exists() else start

repo_root = find_repo_root()
drive_data_dir = shortcut_path / 'nlp'
repo_data_dir = repo_root / 'data' / 'nlp'
data_dir = drive_data_dir if (drive_data_dir / 'nlp.jsonl').exists() else repo_data_dir

artifact_dir = shortcut_path / 'nlp' / 'models' / 'nlp_answer_model'
repo_artifact_dir = repo_root / 'nlp' / 'src' / 'models' / 'nlp_answer_model'

print('repo_root:', repo_root)
print('data_dir:', data_dir)
print('artifact_dir:', artifact_dir)
print('repo_artifact_dir:', repo_artifact_dir)

Mounted at /content/drive
repo_root: /content
data_dir: /content/drive/MyDrive/til26_data/nlp
artifact_dir: /content/drive/MyDrive/til26_data/nlp/models/nlp_answer_model
repo_artifact_dir: /content/nlp/src/models/nlp_answer_model


In [4]:
random.seed(42)

documents_dir = data_dir / 'documents'
qa_path = data_dir / 'nlp.jsonl'

doc_texts = {}
for path in sorted(documents_dir.glob('*.txt')):
    doc_texts[path.stem] = path.read_text(encoding='utf-8', errors='replace')

qa_rows = []
with qa_path.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            qa_rows.append(json.loads(line))

print(f'Loaded {len(doc_texts)} documents and {len(qa_rows)} QA rows')
print(Counter(row.get('difficulty') for row in qa_rows))
print(Counter(row.get('answerable') for row in qa_rows))
qa_rows[:2]

Loaded 296 documents and 883 QA rows
Counter({'L1': 592, 'L2': 291})
Counter({True: 883})


[{'key': 0,
  'difficulty': 'L1',
  'question': 'What penalty was assessed against Halcyon Technical Services in CGC enforcement action EA-76-088?',
  'answer': '4.5 million Credits and mandatory equipment surrender',
  'source_docs': ['DOC-0193'],
  'answerable': True},
 {'key': 1,
  'difficulty': 'L1',
  'question': 'What internal codename did ONE Network Enterprises use for its classified data-sharing arrangement with the CGC Security Directorate?',
  'answer': 'SEASTITCH',
  'source_docs': ['DOC-0045'],
  'answerable': True}]

In [5]:
TOKEN_RE = re.compile(r"\d+(?:\.\d+)?%?|[a-z]+(?:[-'][a-z0-9]+)*", re.I)

def normalize_space(text):
    return re.sub(r'\s+', ' ', text).strip()

def tokens(text):
    return [m.group(0).lower() for m in TOKEN_RE.finditer(text)]

def doc_title(text, doc_name):
    for line in text.splitlines():
        line = normalize_space(line).strip('#* ')
        if line:
            return line[:180]
    return doc_name

def chunk_document(doc_name, text, max_words=190, overlap=45):
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    title = doc_title(text, doc_name)
    paragraphs = [normalize_space(p) for p in re.split(r'\n\s*\n', text) if normalize_space(p)]
    chunks = []
    current = []
    chunk_id = 0

    def flush():
        nonlocal current, chunk_id
        if not current:
            return
        body = ' '.join(current)
        chunk_text = f'{title}\n{body}' if title not in body[:120] else body
        chunks.append({'doc_name': doc_name, 'chunk_id': chunk_id, 'title': title, 'text': chunk_text})
        chunk_id += 1
        current = current[-overlap:] if overlap else []

    for paragraph in paragraphs:
        words = paragraph.split()
        if len(words) > max_words:
            step = max(1, max_words - overlap)
            for start in range(0, len(words), step):
                window = words[start:start + max_words]
                if window:
                    chunks.append({'doc_name': doc_name, 'chunk_id': chunk_id, 'title': title, 'text': f'{title}\n{" ".join(window)}'})
                    chunk_id += 1
            current = []
            continue
        if current and len(current) + len(words) > max_words:
            flush()
        current.extend(words)
    flush()
    return chunks

class BM25:
    def __init__(self, chunks):
        self.chunks = chunks
        self.term_counts = []
        self.lengths = []
        df = Counter()
        for chunk in chunks:
            counts = Counter(tokens(chunk['text']))
            self.term_counts.append(counts)
            self.lengths.append(sum(counts.values()))
            df.update(counts.keys())
        n = max(1, len(chunks))
        self.avgdl = sum(self.lengths) / n
        self.idf = {term: __import__('math').log(1 + (n - freq + 0.5) / (freq + 0.5)) for term, freq in df.items()}

    def retrieve(self, question, k=6):
        q_terms = Counter(tokens(question))
        scores = []
        k1, b = 1.45, 0.72
        for i, counts in enumerate(self.term_counts):
            length = self.lengths[i] or 1
            norm = k1 * (1 - b + b * length / self.avgdl)
            score = 0.0
            for term, q_count in q_terms.items():
                freq = counts.get(term, 0)
                if not freq:
                    continue
                tf = (freq * (k1 + 1)) / (freq + norm)
                score += self.idf.get(term, 0.0) * tf * (1 + 0.15 * (q_count - 1))
            if score > 0:
                scores.append((score, self.chunks[i]))
        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:k]

all_chunks = []
for doc_name, text in doc_texts.items():
    all_chunks.extend(chunk_document(doc_name, text))

bm25 = BM25(all_chunks)
print(f'Built {len(all_chunks)} chunks')

Built 2326 chunks


In [6]:
def retrieval_hit_at_k(rows, k=6):
    hits = 0
    for row in rows:
        gold_docs = set(row.get('source_docs') or [])
        pred_docs = {chunk['doc_name'] for _, chunk in bm25.retrieve(row['question'], k=k)}
        hits += bool(gold_docs & pred_docs)
    return hits / max(1, len(rows))

for k in [1, 3, 6, 10]:
    print(f'hit@{k}: {retrieval_hit_at_k(qa_rows, k):.3f}')

sample = qa_rows[0]
print('\nQuestion:', sample['question'])
print('Answer:', sample['answer'])
for score, chunk in bm25.retrieve(sample['question'], k=3):
    print('\nscore', round(score, 2), chunk['doc_name'], chunk['text'][:500])

hit@1: 0.886
hit@3: 0.959
hit@6: 0.981
hit@10: 0.989

Question: What penalty was assessed against Halcyon Technical Services in CGC enforcement action EA-76-088?
Answer: 4.5 million Credits and mandatory equipment surrender

score 41.77 DOC-0193 # CLAIROS GOVERNANCE COUNCIL ## REGULATORY ENFORCEMENT DIVISION --- **CLASSIFICATION: RESTRICTED** **CGC ENFORCEMENT ACTION EA-76-088** **RE: Unauthorized Satellite Communications Equipment — Orbital Communications Restriction Act of 70 PCE** **Filing Date:** 76-12-04 **Issuing Authority:** CGC Regulatory Enforcement Division **Respondent:** Halcyon Technical Services (Haven Registration No. HTS-2204-09) --- ## SECTION 1 — FILING SUMMARY 1.1 This enforcement action is filed by the CGC Regulatory

score 26.2 DOC-0193 CLAIROS GOVERNANCE COUNCIL
to the CGC. Respondent holds no right of recovery or compensation for surrendered items. 5.2 Respondent is further directed to submit a compliance attestation within ninety (90) days confirming that no add

In [7]:
MAX_CONTEXT_CHARS = 5600
TOP_K = 6

chunks_by_doc = defaultdict(list)
for chunk in all_chunks:
    chunks_by_doc[chunk['doc_name']].append(chunk)

def build_prompt(question, context):
    return (
        'Answer the question using only the context. '
        'Return a concise answer, not a full explanation. '
        'Every question is answerable in the corpus, so provide the best supported answer.\n\n'
        f'Question: {question}\n\n'
        f'Context:\n{context}\n\n'
        'Answer:'
    )

def pack_context(chunks, max_chars=MAX_CONTEXT_CHARS):
    blocks = []
    used = 0
    for chunk in chunks:
        block = f"[{chunk['doc_name']} / CHUNK {chunk['chunk_id']}]\n{chunk['text'].strip()}"
        if used + len(block) > max_chars:
            remaining = max_chars - used
            if remaining <= 400:
                break
            block = block[:remaining]
        blocks.append(block)
        used += len(block)
    return '\n\n'.join(blocks)

def gold_context_for_row(row):
    selected = []
    q_tokens = set(tokens(row['question']))
    for doc_name in row.get('source_docs') or []:
        doc_chunks = chunks_by_doc.get(doc_name, [])
        ranked = sorted(
            doc_chunks,
            key=lambda c: len(q_tokens & set(tokens(c['text']))),
            reverse=True,
        )
        selected.extend(ranked[:3])
    return pack_context(selected)

def retrieved_context_for_question(question):
    return pack_context([chunk for _, chunk in bm25.retrieve(question, k=TOP_K)])

answerable_rows = [row for row in qa_rows if row.get('answerable', True)]
examples = []
for row in answerable_rows:
    answer = row['answer'] if row.get('answer') is not None else ''
    retrieved_context = retrieved_context_for_question(row['question'])
    examples.append({'input': build_prompt(row['question'], retrieved_context), 'target': answer, 'kind': 'retrieved'})

    gold_context = gold_context_for_row(row)
    if gold_context:
        examples.append({'input': build_prompt(row['question'], gold_context), 'target': answer, 'kind': 'gold'})
random.shuffle(examples)
print(Counter(example['kind'] for example in examples))
print('Total examples:', len(examples))
print(examples[0]['input'][:700])
print('TARGET:', repr(examples[0]['target']))

Counter({'retrieved': 883, 'gold': 883})
Total examples: 1766
Answer the question using only the context. Return a concise answer, not a full explanation. Every question is answerable in the corpus, so provide the best supported answer.

Question: What was Genesis Labs originally founded as before the Cascade era?

Context:
[DOC-0264 / CHUNK 0]
# CLAIROS GOVERNANCE COUNCIL ## PERSONNEL DIVISION — CROSS-REFERENCE DOSSIER --- **CLASSIFICATION: CONFIDENTIAL** **Document Reference:** CGC-PD-0264 **Date of Compilation:** 50-06-15 PCE (2092-06-15 CE) **Compiled by:** CGC Personnel Division, Accession Review Unit **Subject:** MULLIN, Cary **Dossier Purpose:** Accession-era assessment of senior leadership of newly inducted CGC member Genesis Labs; for use by C
TARGET: 'a designer-baby clinic'


In [8]:
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

BASE_MODEL = 'google/flan-t5-base'  # or a local checkpoint folder, e.g. shortcut_path / 'nlp/models/flan-t5-base'
USE_ALL_FOR_FINAL_RUN = False
VALIDATION_FRACTION = 0.12
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 128

random.shuffle(examples)
if USE_ALL_FOR_FINAL_RUN:
    train_examples = examples
    val_examples = examples[:min(96, len(examples))]
else:
    split = int(len(examples) * (1 - VALIDATION_FRACTION))
    train_examples = examples[:split]
    val_examples = examples[split:]

train_ds = Dataset.from_list(train_examples)
val_ds = Dataset.from_list(val_examples)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model.config.use_cache = False
model.gradient_checkpointing_enable()

USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
USE_FP16 = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('bf16:', USE_BF16, 'fp16:', USE_FP16)

def preprocess(batch):
    model_inputs = tokenizer(
        batch['input'],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['target'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

non_empty_targets = sum(bool(example['target'].strip()) for example in train_examples)
label_lengths = [len([token for token in item['labels'] if token != tokenizer.pad_token_id]) for item in tokenized_train.select(range(min(20, len(tokenized_train))))]
print(f'train examples: {len(train_examples)}, validation examples: {len(val_examples)}')
print(f'non-empty train targets: {non_empty_targets}/{len(train_examples)}')
print('first target:', repr(train_examples[0]['target']))
print('first decoded labels:', repr(tokenizer.decode([token for token in tokenized_train[0]['labels'] if token != tokenizer.pad_token_id], skip_special_tokens=True)))
print('sample label token lengths:', label_lengths)
assert train_examples, 'No training examples were built.'
assert non_empty_targets > 0, 'All training targets are empty; loss can collapse to 0.'
assert max(label_lengths or [0]) > 0, 'Tokenized labels are empty; check target preprocessing.'

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
model.to(device)
sanity_batch = collator([tokenized_train[i] for i in range(min(2, len(tokenized_train)))])
sanity_batch = {key: value.to(device) for key, value in sanity_batch.items()}
with torch.no_grad():
    sanity_loss = model(**sanity_batch).loss
print('initial forward loss:', float(sanity_loss.detach().cpu()))
assert torch.isfinite(sanity_loss), 'Initial model loss is not finite; restart runtime and check precision/model path.'
assert float(sanity_loss.detach().cpu()) > 0.0, 'Initial model loss is zero; labels are not being used correctly.'

run_dir = shortcut_path / 'nlp' / 'runs' / 'flan_t5_rag'
args = Seq2SeqTrainingArguments(
    output_dir=str(run_dir),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    num_train_epochs=6,
    warmup_ratio=0.06,
    weight_decay=0.01,
    predict_with_generate=False,
    bf16=USE_BF16,
    fp16=USE_FP16,
    max_grad_norm=1.0,
    optim='adamw_torch',
    logging_nan_inf_filter=False,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

train examples: 1554, validation examples: 212
non-empty train targets: 1554/1554
first target: 'Sarento manufacturing level'
first decoded labels: 'Sarento manufacturing level'
sample label token lengths: [6, 4, 5, 14, 10, 3, 2, 4, 8, 5, 27, 4, 11, 11, 20, 5, 3, 36, 22, 13]
initial forward loss: 1.6030781269073486


/tmp/ipykernel_7472/4152321362.py:109: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss,Validation Loss
100,1.502200,1.375029
200,1.186500,1.255337
300,1.067000,1.188677
400,0.972800,1.147929
500,0.939900,1.123163


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=582, training_loss=1.189191938675556, metrics={'train_runtime': 2125.2339, 'train_samples_per_second': 4.387, 'train_steps_per_second': 0.274, 'total_flos': 1.2014824726554624e+16, 'train_loss': 1.189191938675556, 'epoch': 5.947232947232948})

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

def answer_with_current_model(question, k=TOP_K):
    context = retrieved_context_for_question(question)
    prompt = build_prompt(question, context)
    inputs = tokenizer(prompt, return_tensors='pt', max_length=MAX_SOURCE_LENGTH, truncation=True).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=96,
            num_beams=4,
            length_penalty=0.8,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

for row in random.sample(qa_rows, k=8):
    pred = answer_with_current_model(row['question'])
    print('\nQ:', row['question'])
    print('GT:', row['answer'])
    print('PR:', repr(pred))


Q: What are the estimated extractable reserves of mixed ore at Blackshore site BS-17?
GT: 1.4 million metric tons
PR: '1.4 million metric tons'

Q: Approximately how many residents did Haven Island have between the end of its founding decade and the close of its fastest growth period (18-35 PCE)?
GT: approximately 75 million residents
PR: 'approximately 55 million residents'

Q: What is the estimated cost of the structural reinforcement required for berth WH-3 at West Haven Terminal as of 77 PCE?
GT: 780 million Phi Credits
PR: '780 million Phi Credits'

Q: In fiscal year 40 PCE, did Edge Research's combined parent-company support from Phyrexis and Cyanite exceed its own total reported revenue, and by how much?
GT: Yes — combined support exceeded total revenue by 326 billion Phi Credits
PR: 'No.'

Q: What share of Cyanite's Q1 77 PCE operational budget was consumed by Floodwall maintenance?
GT: 34%
PR: '34%'

Q: What did the Wampa security director flag as suspicious about Cyanite Ind

In [15]:
import zipfile
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

eval_zip = data_dir / 'models' / 'nlp_eval.zip'
eval_model_dir = Path('/content/local_data/nlp_eval/nlp_eval')

if eval_zip.exists() and not (eval_model_dir / 'config.json').exists():
    eval_model_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(eval_zip, 'r') as zip_ref:
        zip_ref.extractall(eval_model_dir)

def format_equivalence_input(question, reference, candidate):
    return f'Question: {question} Reference: {reference} Candidate: {candidate}'

from transformers import PreTrainedTokenizerFast

ae_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(eval_model_dir / 'tokenizer.json'),
    unk_token='[UNK]',
    pad_token='[PAD]',
    cls_token='[CLS]',
    sep_token='[SEP]',
    mask_token='[MASK]',
    model_max_length=8192,
)

ae_model = AutoModelForSequenceClassification.from_pretrained(str(eval_model_dir)).to(device)
ae_model.eval()

@torch.no_grad()
def answer_equivalence_scores(rows, batch_size=16, limit=160):
    rows = list(rows)[:limit]
    predictions = [answer_with_current_model(row['question']) for row in rows]
    triples = [
        format_equivalence_input(row['question'], row['answer'] or '', pred)
        for row, pred in zip(rows, predictions)
    ]
    probs = []
    for start in range(0, len(triples), batch_size):
        batch_texts = triples[start:start + batch_size]
        enc = ae_tokenizer(
            batch_texts,
            max_length=128,
            padding=True,
            truncation=True,
            return_tensors='pt',
        ).to(device)
        logits = ae_model(**enc).logits
        probs.extend(F.softmax(logits, dim=-1)[:, 1].detach().cpu().tolist())

    passed = [p >= 0.5 for p in probs]
    print(f'Answer-equivalence pass rate: {sum(passed)}/{len(passed)} = {sum(passed) / max(1, len(passed)):.3f}')
    for row, pred, prob, ok in list(zip(rows, predictions, probs, passed))[:12]:
        mark = 'OK' if ok else 'MISS'
        print('\n', mark, f'prob={prob:.3f}')
        print('Q:', row['question'])
        print('GT:', row['answer'])
        print('PR:', repr(pred))
    return probs, predictions

score_rows = random.sample(answerable_rows, k=min(160, len(answerable_rows)))
ae_probs, ae_predictions = answer_equivalence_scores(score_rows)

/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Answer-equivalence pass rate: 101/160 = 0.631

 MISS prob=0.009
Q: How much greater was cancer incidence among first-generation Somatic Augmentation recipients compared to unaugmented peers of the same age, and roughly how many patients in the cohort does that elevated rate represent?
GT: Cancer incidence was 1.7 percentage points higher (roughly 2.2× the control rate), affecting approximately 146 cohort members.
PR: '3.1%; roughly 4700 patients in the cohort'

 OK prob=1.000
Q: How long did the unidentified survey vessel detected near Aquaculture Zone 7 on 2119-03-14 remain under observation before departing?
GT: 3 hours and 22 minutes
PR: '3 hours and 22 minutes'

 OK prob=1.000
Q: Who published an article mentioning a quote that Cyanite Industries' internal memo prohibits spokespeople from engaging with in 2104?
GT: Renhwa Media
PR: 'Renhwa Media'

 MISS prob=0.000
Q: What legal basis does Dr. Tsukamo's paper identify for challenging the CGC's trade actions against Bloc nations, and

In [16]:
artifact_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(artifact_dir))
tokenizer.save_pretrained(str(artifact_dir))

if (repo_root / 'nlp' / 'src').exists():
    repo_artifact_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(artifact_dir, repo_artifact_dir, dirs_exist_ok=True)
    print('Copied artifacts to repo:', repo_artifact_dir)
else:
    print('Repo path not found. Manually copy this folder before Docker build:')
    print(artifact_dir)

zip_base = artifact_dir.parent / 'nlp_answer_model'
shutil.make_archive(str(zip_base), 'zip', root_dir=artifact_dir)
print('Saved model folder:', artifact_dir)
print('Saved model zip:', str(zip_base) + '.zip')

Repo path not found. Manually copy this folder before Docker build:
/content/drive/MyDrive/til26_data/nlp/models/nlp_answer_model
Saved model folder: /content/drive/MyDrive/til26_data/nlp/models/nlp_answer_model
Saved model zip: /content/drive/MyDrive/til26_data/nlp/models/nlp_answer_model.zip


In [22]:
!ls /content/drive/MyDrive/til26_data/nlp/models/nlp_answer_model

config.json		special_tokens_map.json  tokenizer.json
generation_config.json	spiece.model		 training_args.bin
model.safetensors	tokenizer_config.json
